### Defining Hardware Lookup Table
Since we want to optimize over the latency and VRAM of our model, we'll create a lookup table.  This will record the latency and VRAM of the different building blocks in our supernet model, so we can quickly estimate the hardware performance of a specific path.

In [20]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if os.path.exists(path):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/epoc2/CS6423_knowledge_distillation_project


In [21]:
import torch
import time
import json

def generate_hardware_lut(supernet, search_space):
    supernet.eval().cuda()
    lut = {}  # dictionary where we'll store observed performance
    
    # iterate through all searchable dimensions
    for res in search_space['res']:
        for width in [0.65, 0.8, 1.0, 1.2]:
            for depth in search_space['depth']:
                for exp in search_space['exp']:
                    config = {
                        'res': res, 'width': width, 
                        'depth': [depth]*4, 'exp': [exp]*4
                    }
                    
                    # Measurement logic
                    dummy_input = torch.randn(1, 1, 224, 224).cuda() # dummy data
                    
                    # warm up gpu
                    for _ in range(10): _ = supernet(dummy_input, config)
                    
                    torch.cuda.synchronize()
                    start = time.time()
                    for _ in range(50): _ = supernet(dummy_input, config)
                    torch.cuda.synchronize()
                    
                    latency = (time.time() - start) / 50 * 1000 # ms
                    vram = torch.cuda.max_memory_allocated() / (1024**2) # MB
                    
                    key = f"r{res}_w{width}_d{depth}_e{exp}"
                    lut[key] = {'latency': latency, 'vram': vram}
                    
    with open("hardware_lut.json", "w") as f:
        json.dump(lut, f)
    return lut

In [22]:
import time
import json
import torch

def generate_hardware_lut(supernet, device, save_path="hardware_lut.json"):
    supernet.to(device)
    supernet.eval()  # no more training
    
    # define search space
    resolutions = [128, 160, 190, 224]
    widths = [0.65, 0.8, 1.0, 1.2]
    depths = [2, 3, 4]
    expansions = [3, 4, 6]
    
    lut = {}  # dictionary where we'll save results    
    with torch.no_grad():
        for res in resolutions:
            for w in widths:
                for d in depths:
                    for e in expansions:
                        config = {
                            'res': res,
                            'width': w,
                            'depth': [d, d, d, d], # simplified
                            'exp': [e, e, e, e]
                        }
                        
                        # dummy input with specific resolution
                        dummy_input = torch.randn(1, 3, res, res).to(device)
                        
                        # GPU warmup
                        for _ in range(10):
                            _ = supernet(dummy_input, config)
                        
                        # timing
                        torch.cuda.synchronize()
                        start_time = time.time()
                        
                        iterations = 50
                        for _ in range(iterations):
                            _ = supernet(dummy_input, config)
                            
                        torch.cuda.synchronize()
                        latency = (time.time() - start_time) / iterations * 1000 # ms
                        
                        key = f"res{res}_w{w}_d{d}_e{e}"
                        lut[key] = round(latency, 3)
                        print(f"Config: {key} | Latency: {latency:.2f}ms")

    with open(save_path, 'w') as f:
        json.dump(lut, f, indent=4)
    
    return lut

In [24]:
from knowledge_distillation.model_init import Supernet
width = [0.65, 0.8, 1.0, 1.2]
snet = Supernet(width, num_classes=165)
checkpoint = torch.load('supernet_epoch_100.pth', map_location='cuda')
snet.load_state_dict(checkpoint['state_dict'])
snet.to('cuda')

lookup_table = generate_hardware_lut(snet, 'cuda')


Starting Hardware Benchmarking on cuda...
Config: res128_w0.65_d2_e3 | Latency: 4.70ms
Config: res128_w0.65_d2_e4 | Latency: 8.04ms
Config: res128_w0.65_d2_e6 | Latency: 9.25ms
Config: res128_w0.65_d3_e3 | Latency: 6.82ms
Config: res128_w0.65_d3_e4 | Latency: 11.66ms
Config: res128_w0.65_d3_e6 | Latency: 13.93ms
Config: res128_w0.65_d4_e3 | Latency: 7.36ms
Config: res128_w0.65_d4_e4 | Latency: 12.64ms
Config: res128_w0.65_d4_e6 | Latency: 15.91ms
Config: res128_w0.8_d2_e3 | Latency: 4.13ms
Config: res128_w0.8_d2_e4 | Latency: 11.48ms
Config: res128_w0.8_d2_e6 | Latency: 10.83ms
Config: res128_w0.8_d3_e3 | Latency: 6.25ms
Config: res128_w0.8_d3_e4 | Latency: 16.66ms
Config: res128_w0.8_d3_e6 | Latency: 16.14ms
Config: res128_w0.8_d4_e3 | Latency: 7.12ms
Config: res128_w0.8_d4_e4 | Latency: 17.86ms
Config: res128_w0.8_d4_e6 | Latency: 17.61ms
Config: res128_w1.0_d2_e3 | Latency: 9.28ms
Config: res128_w1.0_d2_e4 | Latency: 16.99ms
Config: res128_w1.0_d2_e6 | Latency: 37.70ms
Config: res12

### Evolutionary Search
From our trained supernet, we want to "evolve" the best student architecture.  We will look to balance accuracy and latency, all within our hardware constraints (which is easy to check thanks to our lookup table).

In [25]:
import random
import json
import torch

# define search space
RES_CHOICES = [128, 160, 190, 224]
WIDTH_CHOICES = [0.65, 0.8, 1.0, 1.2]
DEPTH_CHOICES = [2, 3, 4]
EXP_CHOICES = [3, 4, 6]

def get_random_config():
    # generates random configurations
    return {
        'res': random.choice(RES_CHOICES),
        'width': random.choice(WIDTH_CHOICES),
        'depth': [random.choice(DEPTH_CHOICES) for _ in range(4)],
        'exp': [random.choice(EXP_CHOICES) for _ in range(4)]
    }

In [26]:
def evaluate_fitness(config, supernet, val_loader, lut, latency_constraint, device):
    """
    Calculates the accuracy and checks latency.
    If latency > constraint, we return a fitness of 0.
    """
    # check expected latency in lookup table
    # we use the average depth/exp to match your simplified lookup table
    avg_d = int(round(sum(config['depth']) / 4))
    avg_e = int(round(sum(config['exp']) / 4))
    key = f"res{config['res']}_w{config['width']}_d{avg_d}_e{avg_e}"
    
    latency = lut.get(key, 999) # Default to huge latency if missing
    
    if latency > latency_constraint:
        return 0, latency # Disqualified!

    # check accuracy over a few batches
    supernet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for i, (images, labels) in enumerate(val_loader):
            if i > 10: break
            images, labels = images.to(device), labels.to(device)
            outputs = supernet(images, config)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    return accuracy, latency

In [27]:
def crossover(config_a, config_b):
    # mixes successful parents
    new_config = {}
    # randomly pick resolution and width from either parent
    new_config['res'] = random.choice([config_a['res'], config_b['res']])
    new_config['width'] = random.choice([config_a['width'], config_b['width']])
    
    # mix depths and expansions block by block
    new_config['depth'] = [random.choice([d1, d2]) for d1, d2 in zip(config_a['depth'], config_b['depth'])]
    new_config['exp'] = [random.choice([e1, e2]) for e1, e2 in zip(config_a['exp'], config_b['exp'])]
    
    return new_config

def mutate(config, mutation_rate=0.2):
   # randomly flips genes to explore
    if random.random() < mutation_rate:
        gene_to_flip = random.choice(['res', 'width', 'depth', 'exp'])
        if gene_to_flip == 'res': config['res'] = random.choice(RES_CHOICES)
        elif gene_to_flip == 'width': config['width'] = random.choice(WIDTH_CHOICES)
        elif gene_to_flip == 'depth': config['depth'][random.randint(0,3)] = random.choice(DEPTH_CHOICES)
        elif gene_to_flip == 'exp': config['exp'][random.randint(0,3)] = random.choice(EXP_CHOICES)
    return config

In [28]:
def run_evolutionary_search(supernet, val_loader, lut, device, generations=10, population_size=20, constraint=15.0):
    # initialize population
    population = [get_random_config() for _ in range(population_size)]
    best_overall = None
    
    for gen in range(generations):
        scores = []
        print(f"\n--- Generation {gen} ---")
        
        # evaluate population
        for config in population:
            acc, lat = evaluate_fitness(config, supernet, val_loader, lut, constraint, device)
            scores.append((acc, lat, config))
        
        # sort by accuracy
        scores.sort(key=lambda x: x[0], reverse=True)
        
        # keep track of best model
        if best_overall is None or scores[0][0] > best_overall[0]:
            best_overall = scores[0]
            print(f"NEW BEST: Acc {best_overall[0]:.2f}% | Lat {best_overall[1]:.2f}ms")
            
        # selection, keep top 25% as parents
        parents = [s[2] for s in scores[:population_size // 4]]
        
        # create next generation
        new_population = parents.copy() # Elitism: keep best parents
        while len(new_population) < population_size:
            p1, p2 = random.sample(parents, 2)
            child = crossover(p1, p2)
            child = mutate(child)
            new_population.append(child)
            
        population = new_population

    return best_overall

In [29]:
search_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=4)
run_evolutionary_search(snet, search_loader, lookup_table, 'cuda')


--- Generation 0 ---
NEW BEST: Acc 5.11% | Lat 13.43ms

--- Generation 1 ---
NEW BEST: Acc 6.25% | Lat 9.04ms

--- Generation 2 ---
NEW BEST: Acc 9.94% | Lat 9.04ms

--- Generation 3 ---
NEW BEST: Acc 10.80% | Lat 5.37ms

--- Generation 4 ---

--- Generation 5 ---

--- Generation 6 ---

--- Generation 7 ---

--- Generation 8 ---

--- Generation 9 ---


(10.795454545454545,
 5.369,
 {'res': 190, 'width': 0.65, 'depth': [2, 3, 2, 2], 'exp': [3, 3, 3, 4]})

In [5]:
# define function to initialize and return our pretrained model - the resnet50
def load_pre_model(checkpoint_path, device):
    pre_model = models.resnet50(weights=None)
    pre_model.fc = nn.Linear(2048, 165)
    fc_num_ftrs = pre_model.fc.in_features

    torch.serialization.add_safe_globals([argparse.Namespace])
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    state_dict = checkpoint['model']
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('_orig_mod.', '') 
        new_state_dict[name] = v

    pre_model.load_state_dict(new_state_dict, strict=True)
    
    pre_model.eval()
    for p in pre_model.parameters():
        p.requires_grad = False

    # send to gpu and output summary
    pre_model.to(device)
    # summary(pre_model, input_size=(3, 224, 224))
    return pre_model

In [20]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from knowledge_distillation.model_init import Supernet
from knowledge_distillation.train_loop import DistillationLoss

def fine_tune_student(train_loader, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # winning architecture from GA
    winning_config = {
        'res': 190, 
        'width': 0.65, 
        'depth': [2, 3, 2, 2], 
        'exp': [3, 3, 3, 4]
    }
    
    # initialize student architecture - winning subset of the supernet
    student = Supernet(width_mult_list=[0.65, 1.2], num_classes=165).to(device)
    checkpoint = torch.load('supernet_epoch_100.pth', map_location=device)
    student.load_state_dict(checkpoint['state_dict'])
    print("Student initialized with best params.")

    # initialize teacher
    teacher = load_pre_model('RadImageNet_weights/resnet50.pth', device)
    teacher.eval() # Teacher stays in eval mode
    
    # fine tuninghyperparams
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = DistillationLoss(alpha=0.7, T=2.0)
    epochs = 50 # 50 epochs of fine-tuning
    
    print(f"Starting Fine-tuning for: {winning_config}")
    
    for epoch in range(epochs):
        student.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # forward pass
            with torch.no_grad():
                teacher_logits = teacher(images)
            
            student_logits = student(images, winning_config)
            
            loss = criterion(student_logits, teacher_logits, labels)
            loss.backward()
            
            # gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=2.0)
            optimizer.step()
            
            running_loss += loss.item()
            
        avg_loss = running_loss / len(train_loader)
        scheduler.step()
        
        # validation check
        val_acc = evaluate_standalone(student, winning_config, test_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Test Acc: {val_acc:.2f}%")
        
        if (epoch + 1) % 10 == 0:
            torch.save(student.state_dict(), f"final_student_v2_epoch_{epoch+1}.pth")

def evaluate_standalone(model, config, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images, config)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

if __name__ == "__main__":
    device = 'cuda'
    pretrained_model_checkpoint = 'RadImageNet_weights/resnet50.pth'
    pre_model = load_pre_model(pretrained_model_checkpoint, device)
    train_loader = DataLoader(train_set, batch_size=32, shuffle=False, num_workers=1)
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=1)

    fine_tune_student(train_loader, test_loader)

Student initialized with best params.
Starting Fine-tuning for: {'res': 190, 'width': 0.65, 'depth': [2, 3, 2, 2], 'exp': [3, 3, 3, 4]}
Epoch 1/50 | Loss: 5.7974 | Test Acc: 1.33%
Epoch 2/50 | Loss: 4.9890 | Test Acc: 3.33%
Epoch 3/50 | Loss: 4.1287 | Test Acc: 4.33%
Epoch 4/50 | Loss: 3.5473 | Test Acc: 6.78%
Epoch 5/50 | Loss: 3.1969 | Test Acc: 6.89%
Epoch 6/50 | Loss: 2.9148 | Test Acc: 6.22%
Epoch 7/50 | Loss: 2.6827 | Test Acc: 7.33%
Epoch 8/50 | Loss: 2.4773 | Test Acc: 8.44%
Epoch 9/50 | Loss: 2.3032 | Test Acc: 8.22%
Epoch 10/50 | Loss: 2.1452 | Test Acc: 9.33%
Epoch 11/50 | Loss: 2.0033 | Test Acc: 11.56%
Epoch 12/50 | Loss: 1.8797 | Test Acc: 12.11%
Epoch 13/50 | Loss: 1.7914 | Test Acc: 12.78%
Epoch 14/50 | Loss: 1.7022 | Test Acc: 15.11%
Epoch 15/50 | Loss: 1.6320 | Test Acc: 14.00%
Epoch 16/50 | Loss: 1.5757 | Test Acc: 17.67%
Epoch 17/50 | Loss: 1.5165 | Test Acc: 20.11%
Epoch 18/50 | Loss: 1.4793 | Test Acc: 21.22%
Epoch 19/50 | Loss: 1.4368 | Test Acc: 17.56%
Epoch 20/

KeyboardInterrupt: 

Old Code

In [ ]:
from train_loop import sample_configs
import random

# function to grab accuracy score
def evaluate_accuracy(supernet, config, val_loader):
    supernet.eval()
    correct = 0
    total = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
        
            outputs = supernet(images, config)    
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    return accuracy


# function for breeding the next generation
def create_next_generation(parents, population_size=50, mutation_rate=0.2):
    next_gen = []
    
    # keep parents
    for p in parents:
        next_gen.append(p['config'])
        
    while len(next_gen) < population_size:
        # pick two random parents
        p1 = random.choice(parents)['config']
        p2 = random.choice(parents)['config']
        
        # CROSSOVER - mix traits from both parents
        child = {
            'res': random.choice([p1['res'], p2['res']]),
            'width': random.choice([p1['width'], p2['width']]),
            # swap depth/exp stages
            'depth': [p1['depth'][i] if random.random() > 0.5 else p2['depth'][i] for i in range(4)],
            'exp': [p1['exp'][i] if random.random() > 0.5 else p2['exp'][i] for i in range(4)]
        }
        
        # MUTATE - occasionally flip a trait to a totally random value to explore
        if random.random() < mutation_rate:
            mutation_type = random.choice(['res', 'width', 'depth', 'exp'])
            if mutation_type == 'res':
                child['res'] = random.choice([128, 160, 190, 224])
            elif mutation_type == 'width':
                child['width'] = random.choice([0.65, 0.8, 1.0, 1.2])
            elif mutation_type == 'depth':
                idx = random.randint(0, 3)
                child['depth'][idx] = random.randint(2, 4)
            elif mutation_type == 'exp':
                idx = random.randint(0, 3)
                child['exp'][idx] = random.choice([3, 4, 6])
                
        next_gen.append(child)
        
    return next_gen


In [ ]:
# function to orchestrate evolutionary search
def evolutionary_search(supernet, val_loader, lut, generations=20, population_size=50):
    # initialize random population of configs
    population = sample_configs(population_size)
    pareto_front = []

    for gen in range(generations):
        fitness_scores = []
        
        for config in population:
            # first check hardware estimation from lookup table
            hw_key = f"r{config['res']}_w{config['width']}_d{config['depth'][0]}_e{config['exp'][0]}"
            if lut[hw_key]['vram'] > 5500: # 5.5GB limit
                continue
                
            # evaluate accuracy on validation set
            acc = evaluate_accuracy(supernet, config, val_loader)
            latency = lut[hw_key]['latency']
            
            fitness_scores.append({'config': config, 'acc': acc, 'lat': latency})

        # sort by accuracy and latency
 
       # keep the top 20% and mutate
        fitness_scores.sort(key=lambda x: x['acc'], reverse=True)
        parents = fitness_scores[:10]
        pareto_front.extend(parents)
        
        # crossover & mutation to create next generation
        population = create_next_generation(parents)
        print(f"Gen {gen} Best Acc: {parents[0]['acc']:.2f}%")  # report metrics

    return pareto_front